In [1]:
import accelforge as af
from scheduling.scheduler import *
from af_wrapper import *
import numpy as np


def run(
    einsums,
    compute_units,
    data_dependencies,
    latency_per_component_grid = None,
    total_latency_grid = None,
    actions_grid = None,
    memory_name = None,
    shared_memory_info = None,
):
    schedule, min_latency = best_schedule(
        einsums,
        compute_units,
        shared_memory_info,
        data_dependencies,
        latency_per_component_grid,
        total_latency_grid,
        actions_grid,
        memory_name
    )
    return schedule, min_latency


In [2]:
arch = "arch/eyeriss-dual-core-identical/full.yaml"
workload = "workload/bw-sharing-case-study/mm.yaml"
data_dependencies = {
    "Matmul1": [],
    "Matmul2": []
}
compute_units = ['core1-GlobalBuffer-50', 'core2-GlobalBuffer-50']
einsums = data_dependencies.keys()
memory_name = "MainMemory"

In [11]:
%%capture af_output 
af_baseline = af_map(arch, workload)

Generating pmapping templates for compute MAC2 Einsum Matmul2: 16it [00:00, 90.82it/s]
Generating pmapping templates for compute MAC2 Einsum Matmul1: 16it [00:00, 97.01it/s]
Generating pmapping templates for compute MAC1 Einsum Matmul2: 16it [00:00, 97.39it/s]
Generating pmapping templates for compute MAC1 Einsum Matmul1: 16it [00:00, 96.50it/s]


In [13]:
with open("images/af_baseline-big-2rectmm-eyeriss_dual_identical.svg", "w") as f:
    f.write(af_baseline.render())

In [14]:
af_baseline.latency()


0.030019531026482582

In [15]:
af_baseline.latency()

0.030019531026482582

In [16]:
mems = sum(n for ((unit, act), n) in af_baseline[0].actions(per_component=True).items() if unit == 'MainMemory')
mems

np.float64(100728832.0)

In [17]:
af_baseline.latency()

0.030019531026482582

In [18]:
(mems/af_baseline.latency()) / 6.4e9

np.float64(0.5242880039037086)

In [13]:
%%capture grid_out
# this is for bw-aware version
# RUN TO GENERATE ACCELFORGE VALUES.
# To skip recomputation when running with fresh kernel, comment out
# this cell and use the below assignments instead.

arch = "arch/eyeriss-dual-core-identical/full.yaml"
workload = "workload/bw-sharing-case-study/mm.yaml"

(grid_lats, grid_mems, grid_maps) = af_memoizable_grid(
    einsums, 
    compute_units,
    lambda einsum: "workload/bw-sharing-case-study/"+einsum+".yaml",
    lambda sub_arch: "arch/eyeriss-dual-core-identical/"+sub_arch+".yaml",
    # just_one = True
)

In [3]:
grid_mems = {('core1-GlobalBuffer-50',
  'Matmul1'): {('InputScratchpad1',
   'read'): np.float64(1610612736.0), ('InputScratchpad1', 'write'): np.float32(2.5165824e+07), ('MainMemory',
   'read'): np.float64(50331648.0), ('MainMemory',
   'write'): np.float64(32768.0), ('WeightScratchpad1',
   'read'): np.float64(1610612736.0), ('WeightScratchpad1',
   'write'): np.float32(1.6106127e+09), ('GlobalBuffer',
   'read'): np.float32(3.2768e+06), ('GlobalBuffer',
   'write'): np.float32(524288.0), ('OutputScratchpad1',
   'read'): np.float32(1.7108828e+09), ('OutputScratchpad1',
   'write'): np.float32(1.7108828e+09), ('MAC1',
   'compute'): np.float64(201326592.0)},
 ('core1-GlobalBuffer-50',
  'Matmul2'): {('InputScratchpad1',
   'read'): np.float64(1610612736.0), ('InputScratchpad1',
   'write'): np.float32(2.5165824e+07), ('MainMemory',
   'read'): np.float64(50331648.0), ('MainMemory',
   'write'): np.float64(32768.0), ('WeightScratchpad1',
   'read'): np.float64(1610612736.0), ('WeightScratchpad1',
   'write'): np.float32(1.6106127e+09), ('GlobalBuffer',
   'read'): np.float32(3.2768e+06), ('GlobalBuffer',
   'write'): np.float32(524288.0), ('OutputScratchpad1',
   'read'): np.float32(1.7108828e+09), ('OutputScratchpad1',
   'write'): np.float32(1.7108828e+09), ('MAC1',
   'compute'): np.float64(201326592.0)},
 ('core2-GlobalBuffer-50',
  'Matmul1'): {('InputScratchpad2',
   'read'): np.float64(1610612736.0), ('InputScratchpad2',
   'write'): np.float32(2.5165824e+07), ('MainMemory',
   'read'): np.float64(50331648.0), ('MainMemory',
   'write'): np.float64(32768.0), ('WeightScratchpad2',
   'read'): np.float64(1610612736.0), ('WeightScratchpad2',
   'write'): np.float32(1.6106127e+09), ('GlobalBuffer',
   'read'): np.float32(3.2768e+06), ('GlobalBuffer',
   'write'): np.float32(524288.0), ('OutputScratchpad2',
   'read'): np.float32(1.7108828e+09), ('OutputScratchpad2',
   'write'): np.float32(1.7108828e+09), ('MAC2',
   'compute'): np.float64(201326592.0)},
 ('core2-GlobalBuffer-50',
  'Matmul2'): {('InputScratchpad2',
   'read'): np.float64(1610612736.0), ('InputScratchpad2',
   'write'): np.float32(2.5165824e+07), ('MainMemory',
   'read'): np.float64(50331648.0), ('MainMemory',
   'write'): np.float64(32768.0), ('WeightScratchpad2',
   'read'): np.float64(1610612736.0), ('WeightScratchpad2',
   'write'): np.float32(1.6106127e+09), ('GlobalBuffer',
   'read'): np.float32(3.2768e+06), ('GlobalBuffer',
   'write'): np.float32(524288.0), ('OutputScratchpad2',
   'read'): np.float32(1.7108828e+09), ('OutputScratchpad2',
   'write'): np.float32(1.7108828e+09), ('MAC2',
   'compute'): np.float64(201326592.0)}}

In [4]:
grid_lats = {('core1-GlobalBuffer-50', 'Matmul1'): {'MAC1': np.float32(0.01048576),
  'InputScratchpad1': np.float32(0.0059842104),
  'MainMemory': np.float32(0.00786944),
  'WeightScratchpad1': np.float32(0.0034257055),
  'GlobalBuffer': np.float32(0.00059392),
  'OutputScratchpad1': np.float32(0.012517933)},
 ('core1-GlobalBuffer-50', 'Matmul2'): {'MAC1': np.float32(0.01048576),
  'InputScratchpad1': np.float32(0.0059842104),
  'MainMemory': np.float32(0.00786944),
  'WeightScratchpad1': np.float32(0.0034257055),
  'GlobalBuffer': np.float32(0.00059392),
  'OutputScratchpad1': np.float32(0.012517933)},
 ('core2-GlobalBuffer-50', 'Matmul1'): {'MAC2': np.float32(0.01048576),
  'InputScratchpad2': np.float32(0.0059842104),
  'MainMemory': np.float32(0.00786944),
  'WeightScratchpad2': np.float32(0.0034257055),
  'GlobalBuffer': np.float32(0.00059392),
  'OutputScratchpad2': np.float32(0.012517933)},
 ('core2-GlobalBuffer-50', 'Matmul2'): {'MAC2': np.float32(0.01048576),
  'InputScratchpad2': np.float32(0.0059842104),
  'MainMemory': np.float32(0.00786944),
  'WeightScratchpad2': np.float32(0.0034257055),
  'GlobalBuffer': np.float32(0.00059392),
  'OutputScratchpad2': np.float32(0.012517933)}}

In [5]:
onem = list(grid_maps.values())[0]

(sum(n for ((unit, act), n) in onem.actions(per_component=True).items() if unit == 'MainMemory')/onem.latency()) / 6.4e9


NameError: name 'grid_maps' is not defined

In [15]:
onem.latency()

np.float32(0.012517933)

In [5]:
# The schedule with ONLY 50/50 split of shared memory, with bandwidth utilization considering

schedule, latency = run(
    einsums,
    compute_units,
    data_dependencies,
    latency_per_component_grid = grid_lats,
    actions_grid = grid_mems,
    memory_name = memory_name
)

{'Matmul1': (Matmul1, core1-GlobalBuffer-50, latency=0.012517932802438736), 'Matmul2': (Matmul2, core1-GlobalBuffer-50, latency=0.012517932802438736)}
Chunks:
[{'einsum': 'Matmul2', 'start': 0, 'end': np.float32(0.012517933), 'bwu': np.float32(0.62865335)}]

Chunks:
[{'einsum': 'Matmul2', 'start': 0, 'end': np.float32(0.012517933), 'bwu': np.float32(0.62865335)}, {'einsum': 'Matmul1', 'start': np.float32(0.012517933), 'end': np.float32(0.025035866), 'bwu': np.float32(0.62865335)}]

{'Matmul1': (Matmul1, core1-GlobalBuffer-50, latency=0.012517932802438736), 'Matmul2': (Matmul2, core1-GlobalBuffer-50, latency=0.012517932802438736)}
Chunks:
[{'einsum': 'Matmul1', 'start': 0, 'end': np.float32(0.012517933), 'bwu': np.float32(0.62865335)}]

Chunks:
[{'einsum': 'Matmul1', 'start': 0, 'end': np.float32(0.012517933), 'bwu': np.float32(0.62865335)}, {'einsum': 'Matmul2', 'start': np.float32(0.012517933), 'end': np.float32(0.025035866), 'bwu': np.float32(0.62865335)}]

{'Matmul1': (Matmul1, core

In [6]:
schedule

{(Matmul1, core1-GlobalBuffer-50, latency=0.012517932802438736): 0,
 (Matmul2, core2-GlobalBuffer-50, latency=0.017641500681166322): 0}

In [7]:
latency

np.float64(0.017641500681166322)

In [8]:
# for node in schedule.keys():
#     for (action, count) in grid_mems[(node.compute_assignment, node.einsum_name)].items():
#         if action[0] == memory_name:
#             print(node, action, count)

            
mems = sum(
    sum(
        count
        for (action, count) in grid_mems[(node.compute_assignment, node.einsum_name)].items()
        if action[0] == memory_name
    ) 
    for node in schedule.keys()
)
mems

(Matmul1, core1-GlobalBuffer-50, latency=0.012517932802438736) ('MainMemory', 'read') 50331648.0
(Matmul1, core1-GlobalBuffer-50, latency=0.012517932802438736) ('MainMemory', 'write') 32768.0
(Matmul2, core2-GlobalBuffer-50, latency=0.017641500681166322) ('MainMemory', 'read') 50331648.0
(Matmul2, core2-GlobalBuffer-50, latency=0.017641500681166322) ('MainMemory', 'write') 32768.0


np.float64(100728832.0)

In [10]:
# bwu
(mems/latency) / 6.4e9 # old schedule: np.float64(0.4762258443913986)

np.float64(0.8921508597509781)